# Tratamiento y preparación de datos — versión completa
Evaluación 1: Calidad y Preparación de Datos para IA · Universidad del Bío-Bío

Cubre: duplicados, filtro del target (sin imputar), **saneamiento de inconsistencias de dominio**, **tratamiento de outliers (winsorizing por IQR)**, imputación, normalización de categóricas, Pipeline + ColumnTransformer y **tabla comparativa** original vs. tratado.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

raw = pd.read_csv("rendimiento_academico_evaluacion.csv")

# Metricas del dataset ORIGINAL (para la tabla comparativa)
orig_rows, orig_cols = raw.shape
orig_missing = int(raw.isnull().sum().sum())
orig_dups    = int(raw.duplicated().sum())
print("Original:", raw.shape, "| faltantes:", orig_missing, "| duplicados:", orig_dups)

## Normalización ortográfica de categóricas
Unifica variantes de escritura para que el One-Hot Encoding no genere dummies fantasma.

In [ ]:
def norm_scholarship(s):
    if pd.isna(s): return np.nan
    t = str(s).strip().lower()
    if t in {"si","sí","s"}: return "Sí"
    if t in {"no","n"}:      return "No"
    return str(s).strip()

def norm_program(s):
    if pd.isna(s): return np.nan
    t = str(s).strip().lower().replace("í","i").replace("á","a").replace("ó","o")
    m = {"ingenieria informatica":"Ingeniería Informática","ingenieria industrial":"Ingeniería Industrial",
         "ingenieria civil":"Ingeniería Civil","administracion":"Administración","contador auditor":"Contador Auditor"}
    return m.get(t, str(s).strip())

df = raw.copy()
df["Scholarship"] = df["Scholarship"].map(norm_scholarship)
df["Program"]     = df["Program"].map(norm_program)
print("Scholarship:", sorted(df["Scholarship"].dropna().unique()))
print("Program:", sorted(df["Program"].dropna().unique()))

## Paso 1 — Duplicados exactos + filtro del TARGET
El target `FinalGrade` **no se imputa**: las filas fuera de [1.0, 10.0] se eliminan. 5100 → 5000 → 4996.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)              # 5100 -> 5000
mask_target = df["FinalGrade"].between(1.0, 10.0)
inconsist_target = int((~mask_target).sum())
print("Targets invalidos eliminados:", inconsist_target)
print(df.loc[~mask_target, ["StudentID","FinalGrade"]])
df = df[mask_target].reset_index(drop=True)                   # 5000 -> 4996
print("Filas:", df.shape[0])

## Paso 1b — Saneamiento de inconsistencias de dominio (features)
Valores imposibles según el significado de cada variable se marcan como `NaN` para luego imputarlos.
Justificación (del diagnóstico): un porcentaje no puede ser <0 ni >100; las horas diarias no pueden ser negativas ni superar 24. Esto **distingue un valor inválido de un outlier legítimo**.

In [ ]:
DOMINIO = {
    "Attendance": (0, 100),           # porcentaje
    "AssignmentsCompleted": (0, 100), # porcentaje
    "PreviousGPA": (1, 10),           # escala 1-10
    "SleepHours": (0, 24),            # horas/día
    "InternetHours": (0, 24),         # horas/día
    "Age": (15, 80),                  # edad plausible
    "StudyHours": (0, 168),           # horas/semana
}
inconsist_features = 0
for c, (lo, hi) in DOMINIO.items():
    bad = (~df[c].between(lo, hi)) & df[c].notna()
    inconsist_features += int(bad.sum())
    df.loc[bad, c] = np.nan
print("Inconsistencias de dominio en features (-> NaN):", inconsist_features)

## Paso 2 — Separación limpia de X e y

In [ ]:
num_features = ["Age","StudyHours","Attendance","PreviousGPA",
                "AssignmentsCompleted","SleepHours","InternetHours"]
cat_features = ["Scholarship","Program"]
X = df.drop(columns=["FinalGrade"])   # 10 columnas (StudentID = identificador)
y = df["FinalGrade"]                  # target aislado
print("X:", X.shape, "| y:", y.shape)

## Transformer de outliers — Winsorizing por IQR
Acota (no elimina) los valores extremos a las vallas `Q1−1.5·IQR` y `Q3+1.5·IQR`. Reproducible dentro del Pipeline. No se aplica al target.

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    """Winsoriza cada columna numérica a las vallas de Tukey (aprendidas en fit)."""
    def __init__(self, k=1.5): self.k = k
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        q1 = np.nanpercentile(X, 25, axis=0); q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lo_ = q1 - self.k*iqr; self.hi_ = q3 + self.k*iqr
        return self
    def transform(self, X):
        X = np.asarray(X, float)
        return np.clip(X, self.lo_, self.hi_)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features)   # winsoriza 1:1, conserva nombres

## Paso 3 — Dataset final interpretable (4996 × 11)
Escala original. Imputa (mediana/moda), aplica winsorizing y cuenta los outliers tratados. Sin OHE ni escalado.

In [ ]:
X_interp = X.copy()
mediana = {c: X_interp[c].median() for c in num_features}
for c in num_features: X_interp[c] = X_interp[c].fillna(mediana[c])
for c in cat_features: X_interp[c] = X_interp[c].fillna(X_interp[c].mode()[0])

clipper = IQRClipper(1.5).fit(X_interp[num_features].values)
antes = X_interp[num_features].values.copy()
X_interp[num_features] = clipper.transform(X_interp[num_features].values)
outliers_tratados = int((antes != X_interp[num_features].values).sum())
print("Outliers tratados (valores winsorizados):", outliers_tratados)

for c in num_features: X_interp[c] = X_interp[c].round(2)
df_limpio = X_interp.copy()
df_limpio["FinalGrade"] = y.round(2).values

assert df_limpio.shape == (4996, 11)
assert df_limpio.isnull().sum().sum() == 0
assert df_limpio["FinalGrade"].between(1.0, 10.0).all()
df_limpio.to_csv("rendimiento_academico_limpio.csv", index=False)
print("Exportado:", df_limpio.shape)

## Paso 4 — Matriz de diseño para modelamiento (4996 × 14)
Numéricas: imputación → winsorizing → escalado. Categóricas: imputación → One-Hot. `StudentID` se descarta.

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("outliers", IQRClipper(1.5)),
            ("scaler",  StandardScaler()),
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot",  OneHotEncoder(handle_unknown="ignore")),
        ]), cat_features),
    ],
    remainder="drop"
)
X_model = preprocesador.fit_transform(X)   # y permanece intacta
print("Matriz de diseño:", X_model.shape)  # (4996, 14)
print(list(preprocesador.get_feature_names_out()))

## Paso 5 — Tabla comparativa: dataset original vs. tratado

In [ ]:
tabla = pd.DataFrame({
    "Indicador": ["Cantidad de registros","Cantidad de variables","Valores faltantes",
                  "Registros duplicados","Inconsistencias","Outliers tratados"],
    "Dataset original": [orig_rows, orig_cols, orig_missing, orig_dups,
                         inconsist_target + inconsist_features, 0],
    "Dataset tratado":  [df_limpio.shape[0], df_limpio.shape[1], 0, 0, 0, outliers_tratados],
})
tabla